# Multi-Provider LLM Client — Demo

This notebook demonstrates the `llm_client` module:
1. Same prompt sent to all 3 providers with a comparative table.
2. Structured output with a Pydantic schema.
3. Streaming with one provider.

> **Prerequisites**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GEMINI_API_KEY` in `../../.env` (repo root).

In [ ]:
import sys
from pathlib import Path

# Ensure the package is importable when running from 04_multi_provider/
sys.path.insert(0, str(Path.cwd()))

from llm_client import (
    AnthropicProvider,
    GeminiProvider,
    LLMResponse,
    OpenAIProvider,
    StreamingResponse,
)

## 1 — Same prompt, all 3 providers

In [ ]:
PROMPT = "Explain what a neural network is in 2 sentences."

openai = OpenAIProvider()
anthropic = AnthropicProvider()
gemini = GeminiProvider()

results: list[LLMResponse] = []

r_openai = await openai.generate(prompt=PROMPT, model="gpt-5.4")
r_anthropic = await anthropic.generate(prompt=PROMPT, model="claude-sonnet-4-6")
r_gemini = await gemini.generate(prompt=PROMPT, model="gemini-3.1-pro")

results = [r_openai, r_anthropic, r_gemini]
print("Calls complete.")

In [ ]:
import pandas as pd

rows = [
    {
        "provider": r.provider,
        "model": r.model,
        "response_text": r.text[:120] + "..." if len(r.text) > 120 else r.text,
        "input_tokens": r.input_tokens,
        "output_tokens": r.output_tokens,
        "cost_usd": f"${r.cost_usd:.6f}",
        "latency_ms": r.latency_ms,
    }
    for r in results
]

df = pd.DataFrame(rows)
df

## 2 — Structured output with a Pydantic schema

In [ ]:
from pydantic import BaseModel


class NeuralNetworkSummary(BaseModel):
    """Structured summary of the neural network explanation."""

    definition: str
    analogy: str
    use_case: str


STRUCTURED_PROMPT = (
    "Explain what a neural network is. "
    "Provide: a one-sentence definition, an analogy, and a real-world use case."
)

structured_result = await anthropic.generate(
    prompt=STRUCTURED_PROMPT,
    model="claude-sonnet-4-6",
    schema=NeuralNetworkSummary,
)

print("Raw text:", structured_result.text[:200])
print()
print("Parsed object:", structured_result.parsed)
print()
if structured_result.parsed:
    s: NeuralNetworkSummary = structured_result.parsed
    print(f"Definition : {s.definition}")
    print(f"Analogy    : {s.analogy}")
    print(f"Use case   : {s.use_case}")

## 3 — Streaming with OpenAI

In [ ]:
import sys

stream_result = await openai.generate(
    prompt=PROMPT,
    model="gpt-5.4",
    stream=True,
)

assert isinstance(stream_result, StreamingResponse)

print("Streaming response (token by token):")
print("-" * 50)

async for chunk in stream_result:
    print(chunk, end="", flush=True)

print()
print("-" * 50)

final = stream_result.final_response
print(f"\nFinal response assembled: {len(final.text)} chars")